In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import concatenate
from keras.utils import plot_model
import math
from tensorflow.keras.layers import Reshape
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import LSTM

In [ ]:
#df = pd.read_pickle("Data/df_preprocessed_train.pickle")
#df_train_original = pd.read_pickle("Data/df_preprocessed_no_encoding_train.pickle")
#df_test_original = pd.read_pickle("Data/df_preprocessed_no_encoding_test.pickle")

In [ ]:
df_train_original = pd.read_pickle("Data/df_preprocessed_train.pickle").reset_index(drop = True)
df_test_original = pd.read_pickle("Data/df_preprocessed_test.pickle").reset_index(drop = True)

In [ ]:
#let's get the original texts back and add an NLP model
with open("Data/df_nlp_processed", "rb") as f:
    df = pickle.load(f)

In [ ]:
df_text_cols = pd.DataFrame()

In [ ]:
df_text_cols[text_embedding_columns] = df[text_embedding_columns]
df_text_cols.set_index(df["original index"])

In [ ]:
len(df_text_cols)

In [ ]:
df_text_cols.head()

In [ ]:
df_train = copy.deepcopy(df_train_original
df_test = copy.deepcopy(df_test_original)

In [ ]:
#let's one-hot-encode for procedure, type contract, central purchasing, eu_fund, framework or dynamic purchasing, 
base_n_encoder_cols = ["cpv_code", "country", "language"]
one_hot_columns = ["type_contract", "procedure", "joint_procurement_involved", "central_purchasing", "eu_fund", "framework or dynamic purchasing", "contracting authority type", "contracting authority activity"]
numerical_columns = ["duration", "nb_tenders_received", "nb_tenders_received_sme"]
text_embedding_columns = ["short_descr", "ac criteria", "object_contract/title", "object descr titles", "ac cost criteria"]
categorical_columns_original = base_n_encoder_cols + one_hot_columns

In [ ]:
#get the text data in a usable format
X_train_text = []
for i in df_train.index:
    row_array = []
    for col in text_embedding_columns:
        text_array = df_train[col].iloc[i]
        row_array.append(np.array(text_array))
    
    X_train_text.append(np.array(row_array))
X_train_text = np.array(X_train_text)

X_test_text = []
for index in df_test.index:
    row_array = []
    for col in text_embedding_columns:
        text_array = df_test[col].iloc[index]
        row_array.append(text_array)
    
    X_test_text.append(row_array)
X_test_text = np.array(X_test_text)

In [ ]:
#initialize the numerical and categorical data
drop_columns = ["original index", "date_conclusion_contract"] + text_embedding_columns
#drop relevant columns
df_train = df_train.drop(columns = drop_columns)
df_test = df_test.drop(columns = drop_columns)
#instantiate the training set for num and cat
X_train = df_train[[col for col in df_train.columns if col != "award_contract/val_total: 0"]].to_numpy()
y_train = df_train["award_contract/val_total: 0"].to_numpy().reshape(-1,1)
#instantiate the test set for num and cat
X_test = df_test[[col for col in df_train.columns if col != "award_contract/val_total: 0"]].to_numpy()
y_test = df_test["award_contract/val_total: 0"].to_numpy().reshape(-1,1)
#in case we need it
x_total = np.append(X_train, X_test)

In [ ]:
print(X_train.shape, X_train_text.shape, y_test.shape, y_train.shape)

In [ ]:
#def create_mlp(dim, regress = False):
#let's build a sequential model for our numerical and categorical data
#input_dim_num_cat = len(X_train[0])
#model_num_cat = Sequential()
#model_num_cat.add(Dense(64, input_dim = (input_dim_num_cat), activation = "relu"))
#model_num_cat.add(Dense(64, activation = "relu"))
#model_num_cat.add(Dense(5, activation = "relu"))
#model_num_cat.add(Flatten())
#model_num_cat.compile()

In [ ]:
input_dim_num_cat = len(X_train[0])
input_num_cat = Input(shape=input_dim_num_cat)
x = Dense(128, activation = "relu")(input_num_cat)
x = Dense(64, activation="relu")(x)
x = Dense(32, activation="relu")(x)
x = Dense(4, activation="relu")(x)
model_num_cat = Model(input_num_cat, x)

In [ ]:
input_dim_text = X_test_text.shape[1:3]
input_text = Input(shape=(input_dim_text))
y = LSTM(units=384, return_sequences=True)(input_text)
y = Dropout(0.1)(y)
y = Dense(units = 128, activation = "sigmoid")(y)
y = LSTM(units = 32, return_sequences=True)(y)
y = Dense(units = 4, activation = "sigmoid")(y)
y = Flatten()(y)
model_text = Model(input_text, y)

In [ ]:
#width, depth = 5, 384

#input_text = Input(shape=(width,depth))
#hidden_1 = Dense(384, activation="relu")(input_text)
#hidden_2 = Dense(120, activation="relu")(hidden_1)
#hidden_3 = Dense(60, activation="relu")(hidden_2)
#hidden_4 = Dense(20, activation = "relu")(hidden_3)
#flatten = Flatten()(hidden_4)
#model_text = Model(input_text, flatten) 

In [ ]:
#combine the models output
combined_input = concatenate([model_text.output, model_num_cat.output])
x = Dense(4, activation="relu")(combined_input)
x = Dense(1, activation="linear")(x)
#create a new layer with the combined output of text,cat and num features
model = Model(inputs=[input_num_cat, input_text], outputs = x)

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=0.01)
loss_function = "mean_absolute_error"
model.compile(loss=loss_function, optimizer=optimizer)
model.fit(
    x=[X_train, X_train_text], y=y_train,
    validation_data=([X_test, X_test_text], y_test),
    epochs=100, batch_size = 32)